# Construindo uma Rede Neural

### Cenário
Uma empresa de e-commerce deseja estimar a **probabilidade de atraso de uma entrega**.

Cada pedido será representado por três características já normalizadas:

- `distancia`
- `quantidade_itens`
- `frete`

O alvo é:

\
y =
\begin{cases}
0 & \text{entrega no prazo}\\
1 & \text{entrega atrasada}
\end{cases}


### Objetivo

Ao longo desta atividade, você irá construir uma pequena rede neural para classificação binária só utilizando tensorflow.

Ao final, deverá compreender o fluxo:

> **Regra da aula:** não utilize `model.fit()`. Hoje queremos entender como a rede **produz uma predição**. O treinamento será estudado na próxima aula.

## Preparação

Execute a célula abaixo para importar as bibliotecas necessárias.

In [1]:
import tensorflow as tf
import numpy as np

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


# Tarefa 1: Representando os dados

Considere os seguintes pedidos:

| Pedido | Distância | Quantidade de itens | Frete | Atrasou? |
|---|---:|---:|---:|---:|
| 1 | 0.20 | 0.10 | 0.30 | 0 |
| 2 | 0.80 | 0.70 | 0.90 | 1 |
| 3 | 0.30 | 0.20 | 0.40 | 0 |
| 4 | 0.90 | 0.80 | 0.70 | 1 |
| 5 | 0.15 | 0.30 | 0.20 | 0 |
| 6 | 0.75 | 0.60 | 0.85 | 1 |

### Sua tarefa

Crie:

- um tensor `X` contendo as três features;
- um tensor `y` contendo o alvo.

Utilize `dtype=tf.float32`.

Depois, exiba os shapes de `X` e `y`.

### Antes de programar

Responda:

**Qual deve ser o shape de `X`?**

Resposta: **(6, 3)**, 6 pedidos com 3 features cada.

**Qual deve ser o shape de `y`?**

Resposta: **(6, 1)**, um alvo por pedido, no mesmo formato da saída da rede.


In [2]:
# Cada linha é um pedido: [distancia, quantidade_itens, frete]
X = tf.constant(
    [
        [0.20, 0.10, 0.30],
        [0.80, 0.70, 0.90],
        [0.30, 0.20, 0.40],
        [0.90, 0.80, 0.70],
        [0.15, 0.30, 0.20],
        [0.75, 0.60, 0.85],
    ],
    dtype=tf.float32,
)

# Alvo em formato coluna (6, 1), o mesmo shape da saída da rede
y = tf.constant([[0], [1], [0], [1], [0], [1]], dtype=tf.float32)

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

Shape de X: (6, 3)
Shape de y: (6, 1)


### Verificação conceitual

Complete:

Cada **linha** de `X` representa **um pedido**.

Cada **coluna** de `X` representa **uma característica (feature)**: distância, quantidade de itens ou frete.

O valor `y = 1` significa **que a entrega atrasou** (classe positiva).


# Tarefa 2: O que acontece dentro de um neurônio?

Antes de construir a rede, vamos lembrar a operação fundamental de um neurônio:

$$
\boxed{z = \sum_i x_i w_i + b}
$$

Considere o primeiro pedido:

$$
x = [0.20,\ 0.10,\ 0.30]
$$

e os seguintes parâmetros:

$$
w = [0.5,\ -0.3,\ 0.8]
$$

$$
b = 0.1
$$

### Primeiro calcule manualmente

Complete a expressão:

$$
z = (0.20 \times 0.5)
  + (0.10 \times -0.3)
  + (0.30 \times 0.8)
  + 0.1
$$

Resultado esperado por você:

$$
\boxed{z = 0.10 - 0.03 + 0.24 + 0.1 = 0.41}
$$

### Depois confirme utilizando TensorFlow.

In [3]:
# Pesos e bias já definidos para o exercício
w = tf.constant([0.5, -0.3, 0.8], dtype=tf.float32)
b = tf.constant(0.1, dtype=tf.float32)

# Selecione o primeiro pedido
x = X[0]

# Calcule z = soma(x * w) + b
z = tf.reduce_sum(x * w) + b

print("z =", round(float(z), 4))

z = 0.41


### Pergunta

O valor de \(z\) representa diretamente a probabilidade de atraso?

**( ) Sim**

**(X) Não**

Justifique:

$z$ é só uma combinação linear das entradas e pode ser qualquer número real, inclusive negativo ou maior que 1. Para virar probabilidade (entre 0 e 1) ainda precisa passar por uma função como a sigmoid.


# Tarefa 3: Função de ativação

Até agora, o neurônio calculou uma combinação das entradas:

$$
z = \sum_i x_iw_i + b
$$

Agora precisamos aplicar uma **função de ativação**.

Nesta tarefa, utilizaremos a **ReLU (Rectified Linear Unit)**:

$$
\boxed{\operatorname{ReLU}(z)=\max(0,z)}
$$

Isso significa que a ReLU compara o valor de $z$ com zero e retorna o **maior deles**:

- se $z < 0$, a saída será **0**;
- se $z > 0$, a saída será o **próprio $z$**.

### Antes de executar o código

Complete mentalmente a tabela:

| $z$ | $\operatorname{ReLU}(z)$ |
|---:|---:|
| -3 | 0 |
| -0.5 | 0 |
| 0 | 0 |
| 2 | 2 |
| 5 | 5 |

### Agora confirme utilizando TensorFlow

Aplique a função ReLU aos mesmos valores e compare o resultado com suas respostas.

In [4]:
valores = tf.constant([-3.0, -0.5, 0.0, 2.0, 5.0])

# Aplique ReLU
resultado = tf.nn.relu(valores)

print(resultado.numpy())

[0. 0. 0. 2. 5.]


### Pense antes de continuar

Por que uma rede neural precisa de funções de ativação **não lineares**?

Porque a composição de camadas lineares continua sendo uma única transformação linear: sem ativação não linear, empilhar camadas equivale a ter uma só, e a rede só consegue separar as classes com uma reta (hiperplano).

A não linearidade (ReLU, sigmoid...) permite que a rede aprenda fronteiras de decisão curvas e relações mais complexas entre as features.


# Tarefa 4: Projetando a arquitetura

Agora vamos construir uma **MLP (Multilayer Perceptron)** para o nosso problema de classificação.

Cada pedido possui **3 características de entrada**:

- distância;
- quantidade de itens;
- valor do frete.

Nossa rede terá a seguinte arquitetura:

$$
\boxed{3 \rightarrow 4 \rightarrow 1}
$$

Podemos interpretá-la como:

$$
\text{3 entradas}
\rightarrow
\text{4 neurônios}
\rightarrow
\text{1 saída}
$$

A configuração será:

| Camada | Quantidade | Função de ativação |
|---|---:|---|
| Entrada | 3 valores | — |
| Camada escondida | 4 neurônios | ReLU |
| Saída | 1 neurônio | Sigmoid |


### Pense no fluxo

Complete:

$$
X
\rightarrow
\boxed{\text{Dense}(4)}
\rightarrow
\boxed{\text{ReLU}}
\rightarrow
\boxed{\text{Dense}(1)}
\rightarrow
\boxed{\text{Sigmoid}}
\rightarrow
\hat{y}
$$

Agora vamos transformar essa arquitetura em código utilizando TensorFlow.

### Agora construa a rede

Use:

- `tf.keras.Sequential`
- `tf.keras.layers.Input`
- `tf.keras.layers.Dense`

> **Dica:** a primeira camada `Dense` deve ter 4 neurônios e a segunda deve ter 1.


In [5]:
# Semente fixa para os pesos iniciais serem sempre os mesmos
tf.random.set_seed(42)

model = tf.keras.Sequential([
    # entrada: 3 features por pedido
    tf.keras.layers.Input(shape=(3,)),

    # camada escondida: 4 neurônios com ReLU
    tf.keras.layers.Dense(4, activation="relu"),

    # camada de saída: 1 neurônio com sigmoid (probabilidade de atraso)
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

# Tarefa 5: Quantos parâmetros existem?

Nossa rede possui a seguinte arquitetura:

$$
\boxed{3 \rightarrow 4 \rightarrow 1}
$$

Antes de executar `model.summary()`, vamos descobrir **manualmente** quantos parâmetros a rede precisa aprender.

Em uma camada `Dense`, cada neurônio possui:

- um **peso para cada entrada**;
- um **bias próprio**.

Assim, podemos calcular:

$$
\text{Pesos} = n_{in} \times n_{out}
$$

$$
\text{Biases} = n_{out}
$$

Portanto:

$$
\boxed{
\text{Parâmetros} =
(n_{in} \times n_{out}) + n_{out}
}
$$

---

## 1. Camada escondida

A camada escondida recebe **3 valores** e possui **4 neurônios**.

Complete:

$$
n_{in} = 3
$$

$$
n_{out} = 4
$$

### Quantos pesos existem?

$$
3 \times 4 = 12
$$

### Quantos biases existem?

$$
4
$$

### Total de parâmetros da camada escondida

$$
12 + 4 = \boxed{16}
$$

---

## 2. Camada de saída

A camada de saída recebe os valores produzidos pelos **4 neurônios da camada anterior** e possui **1 neurônio**.

Complete:

$$
n_{in} = 4
$$

$$
n_{out} = 1
$$

### Quantos pesos existem?

$$
4 \times 1 = 4
$$

### Quantos biases existem?

$$
1
$$

### Total de parâmetros da camada de saída

$$
4 + 1 = \boxed{5}
$$

---

## 3. Total da rede

Some os parâmetros das duas camadas:

$$
\boxed{
\text{Total de parâmetros} =
16 + 5 =
21
}
$$

Agora execute `model.summary()` e verifique se o resultado encontrado pela rede é igual ao seu cálculo.

In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 4)              │            16 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21 (84.00 B)

 Trainable params: 21 (84.00 B)

 Non-trainable params: 0 (0.00 B)

### Conferência

Seu cálculo coincidiu com o TensorFlow?

**(X) Sim**

**( ) Não**

Se não, identifique onde ocorreu a diferença:

Não houve diferença: o `model.summary()` mostra 16 parâmetros na camada escondida, 5 na de saída e 21 no total, igual ao cálculo manual.


# Tarefa 6: Forward Pass

A arquitetura da nossa rede está pronta. Agora vamos **passar os dados pela rede** e observar o que ela produz.

Esse processo é chamado de **forward pass** (propagação para frente).

Durante o forward pass, os dados percorrem a rede da entrada até a saída:

$$
X
\rightarrow
\boxed{\text{Dense}(4)}
\rightarrow
\boxed{\text{ReLU}}
\rightarrow
\boxed{\text{Dense}(1)}
\rightarrow
\boxed{\text{Sigmoid}}
\rightarrow
\hat{y}
$$

Em outras palavras:

1. a rede recebe as características de cada pedido;
2. a camada escondida calcula combinações usando **pesos e biases**;
3. a **ReLU** é aplicada;
4. a camada de saída realiza uma nova combinação;
5. a **sigmoid** transforma o resultado em um valor entre 0 e 1.

O resultado final será:

$$
\hat{y}
$$

onde $\hat{y}$ representa a **probabilidade estimada pelo modelo** de o pedido pertencer à classe positiva (`Atrasou = 1`).

---

## Sua tarefa

Passe o tensor `X` pelo modelo e armazene o resultado em:

```python
y_pred

In [7]:
# Forward pass: os pesos ainda são os da inicialização (sem model.fit)
y_pred = model(X)

print(y_pred.numpy())

[[0.4972061 ]
 [0.47439927]
 [0.4934016 ]
 [0.48431894]
 [0.47890645]
 [0.48074055]]


# Tarefa 7: Probabilidade não é classe

Após o **forward pass**, o neurônio de saída utiliza a função **sigmoid**.

Por isso, a saída da rede é um valor entre 0 e 1:

$$
0 \leq \hat{y} \leq 1
$$

Esse valor pode ser interpretado como a **probabilidade estimada de o pedido pertencer à classe positiva**:

$$
\hat{y} = P(y=1 \mid X)
$$

Suponha que, para um determinado pedido, a rede produza:

$$
\boxed{\hat{y}=0.78}
$$

---

## A. O que significa $\hat{y}=0.78$?

> Resposta: a rede estima **78% de probabilidade** de o pedido pertencer à classe positiva, ou seja, de a entrega atrasar. Ainda não é uma classe, é uma probabilidade.

---

## B. Transformando probabilidade em classe

Para obter uma classe, precisamos definir um **threshold (limiar de decisão)**.

Considere inicialmente:

$$
\text{threshold}=0.5
$$

Utilizaremos a seguinte regra:

$$
\hat{y} \geq 0.5
\Rightarrow
\text{classe }1
$$

$$
\hat{y} < 0.5
\Rightarrow
\text{classe }0
$$

Para $\hat{y}=0.78$, qual classe será prevista?

$$
\boxed{\text{classe}=1}
$$

---

## C. E se alterarmos o threshold?

Agora considere:

$$
\text{threshold}=0.85
$$

A probabilidade produzida pela rede continua sendo:

$$
\hat{y}=0.78
$$

Qual classe será prevista agora?

$$
\boxed{\text{classe}=0}
$$

---

## D. A rede mudou?

Ao alterar o threshold de $0.5$ para $0.85$, os pesos ou biases da rede foram modificados?

- [ ] Sim
- [x] Não

Explique:

> O threshold é uma regra de decisão aplicada **depois** da saída da rede. Os pesos e biases continuam os mesmos e a probabilidade continua 0.78; só muda a régua usada para transformá-la em classe (0.78 < 0.85, então vira classe 0).

---

## Complete o fluxo

$$
X
\rightarrow
\text{Rede Neural}
\rightarrow
\boxed{\hat{y}=0.78}
\rightarrow
\text{Threshold}
\rightarrow
\boxed{\text{Classe}}
$$

**Pergunta final:** em qual etapa a rede neural termina e em qual etapa começa a regra de decisão?

> A rede neural termina quando a sigmoid entrega $\hat{y}$ (a probabilidade). A regra de decisão começa no threshold, que compara $\hat{y}$ com o limiar e decide a classe.

## Aplicando um threshold

Agora transforme as probabilidades da sua rede em classes usando threshold `0.5`.

In [8]:
threshold = 0.5

y_class = tf.cast(y_pred >= threshold, tf.int32)

print("Probabilidades:")
print(y_pred.numpy())

print("\nClasses:")
print(y_class.numpy())

Probabilidades:
[[0.4972061 ]
 [0.47439927]
 [0.4934016 ]
 [0.48431894]
 [0.47890645]
 [0.48074055]]

Classes:
[[0]
 [0]
 [0]
 [0]
 [0]
 [0]]


# Tarefa 8: Comparando previsão e realidade

Agora exiba, para cada pedido:

- valor real;
- probabilidade prevista;
- classe prevista.

Complete o código.


In [9]:
for i in range(len(y)):
    real = int(y[i, 0])
    probabilidade = float(y_pred[i, 0])
    classe = int(y_class[i, 0])

    print(
        f"Pedido {i+1}: "
        f"real={real} | "
        f"probabilidade={probabilidade:.3f} | "
        f"classe={classe}"
    )

Pedido 1: real=0 | probabilidade=0.497 | classe=0
Pedido 2: real=1 | probabilidade=0.474 | classe=0
Pedido 3: real=0 | probabilidade=0.493 | classe=0
Pedido 4: real=1 | probabilidade=0.484 | classe=0
Pedido 5: real=0 | probabilidade=0.479 | classe=0
Pedido 6: real=1 | probabilidade=0.481 | classe=0


### Observação

Como a rede **ainda não foi treinada** (os pesos são os da inicialização aleatória), todas as probabilidades ficam perto de 0.5 e, com threshold 0.5, todos os pedidos viram classe 0. A rede acerta só os 3 pedidos no prazo, ou seja, 50% de acerto: o mesmo que chutar.


# Tarefa 9: Como medir o erro?

Até agora, nossa rede produziu probabilidades.

Mas surge uma pergunta importante:

> **Como podemos medir o quanto uma previsão está errada?**

Considere um pedido cujo valor verdadeiro é:

$$
\boxed{y=1}
$$

Ou seja, esse pedido **realmente atrasou**.

Agora imagine que três modelos produziram as seguintes probabilidades:

| Modelo | Probabilidade prevista |
|---|---:|
| A | $\hat{y}_A = 0.95$ |
| B | $\hat{y}_B = 0.60$ |
| C | $\hat{y}_C = 0.05$ |

---

## Antes de calcular qualquer coisa

### A. Qual modelo produziu a melhor previsão?

> Resposta: o **modelo A** ($\hat{y}_A = 0.95$).

### B. Qual modelo produziu a pior previsão?

> Resposta: o **modelo C** ($\hat{y}_C = 0.05$).

### C. Qual modelo deveria receber a maior penalização?

> Resposta: o **modelo C**.

### D. Explique seu raciocínio

> Como o pedido realmente atrasou ($y=1$), quanto mais perto de 1 melhor a previsão. O modelo A acertou com confiança; o B acertou com pouca confiança; o C deu 95% de chance para a classe errada, um erro confiante, que deve ser o mais penalizado.

---

## Pense sobre o problema

O valor verdadeiro é:

$$
y=1
$$

Portanto, quanto mais a previsão $\hat{y}$ estiver próxima de **1**, melhor deverá ser a previsão.

Mas queremos transformar essa ideia em um **número que represente o erro do modelo**.

Esse número é chamado de:

$$
\boxed{\text{Loss}}
$$

Uma boa função de Loss deve produzir:

$$
\text{boa previsão}
\Rightarrow
\text{Loss pequena}
$$

$$
\text{previsão ruim}
\Rightarrow
\text{Loss grande}
$$

Na classificação binária, uma das funções mais utilizadas para isso é a **Binary Cross-Entropy (BCE)**.

Na próxima etapa, vamos calcular a Loss de cada uma dessas previsões.

## Binary Cross-Entropy

Para problemas de **classificação binária**, uma função de Loss muito utilizada é a **Binary Cross-Entropy (BCE)**.

A ideia é simples:

> **Quanto maior a probabilidade atribuída à resposta correta, menor será a Loss.**

A fórmula é:

$$
\boxed{
L = -\left[
y\log(\hat{y}) +
(1-y)\log(1-\hat{y})
\right]
}
$$

onde:

- $y$ é o **valor verdadeiro** (`0` ou `1`);
- $\hat{y}$ é a **probabilidade prevista pelo modelo**;
- $L$ é a **Loss** da previsão.

---

### No nosso exemplo

Sabemos que:

$$
y=1
$$

Substituindo $y=1$ na fórmula:

$$
L = -\left[
1\log(\hat{y}) +
(1-1)\log(1-\hat{y})
\right]
$$

Como $(1-1)=0$:

$$
\boxed{L=-\log(\hat{y})}
$$

Portanto:

- $\hat{y}$ próximo de **1** → Loss pequena;
- $\hat{y}$ próximo de **0** → Loss grande.

Agora vamos verificar essa intuição utilizando `BinaryCrossentropy` do TensorFlow.

In [10]:
loss_fn = tf.keras.losses.BinaryCrossentropy()

y_true = tf.constant([[1.0]])

predicoes = [
    tf.constant([[0.95]]),
    tf.constant([[0.60]]),
    tf.constant([[0.05]])
]

# Calcule a loss para cada previsão

for predicao in predicoes:
    loss = loss_fn(y_true, predicao)
    print(
        "Predição:",
        float(predicao.numpy()[0, 0]),
        "| Loss:",
        round(float(loss), 4)
    )

Predição: 0.949999988079071 | Loss: 0.0513
Predição: 0.6000000238418579 | Loss: 0.5108
Predição: 0.05000000074505806 | Loss: 2.9957


### Interpretação

Complete:

Quanto **melhor** a previsão, geralmente **menor** será a loss.

Quanto mais **errada e confiante** a previsão, geralmente **maior** será a loss.


# Tarefa 10: Loss da nossa rede

Agora utilize:

- os valores verdadeiros `y`;
- as probabilidades `y_pred`;

para calcular a Binary Cross-Entropy da nossa rede.

> **Importante:** utilize `y_pred`, e não as classes obtidas após o threshold.


In [11]:
# Calcule a loss da rede (usando as probabilidades, não as classes)

loss = loss_fn(y, y_pred)

print("Loss:", round(float(loss), 4))

Loss: 0.7038


### Observação

A loss da rede ficou em torno de **0.70**, muito próxima de $\ln 2 \approx 0.693$, que é a BCE de um modelo que responde 0.5 para tudo. Isso confirma que a rede ainda não aprendeu nada: o treinamento (próxima aula) deve ajustar pesos e biases para reduzir esse valor.


# Desafio final: Como a rede aprende?

Até agora, nossa rede já consegue:

1. receber os dados de entrada;
2. realizar transformações;
3. produzir probabilidades;
4. comparar as previsões com os valores reais;
5. calcular a **Loss**.

Podemos representar o processo atual como:

$$
X
\rightarrow
\text{Rede Neural}
\rightarrow
\hat{y}
\rightarrow
\text{Loss}
$$

Mas existe um problema:

> **Os pesos e biases da nossa rede ainda não foram aprendidos.**

Eles foram apenas **inicializados**.

Para que a rede aprenda, precisamos encontrar uma maneira de ajustar seus parâmetros para produzir previsões melhores.

Isso nos leva à pergunta principal:

$$
\boxed{\text{Como podemos diminuir a Loss?}}
$$

---

## Pense antes da próxima aula

### 1. Como a rede poderia descobrir se um peso deve aumentar ou diminuir?

> Calculando o **gradiente** da loss em relação a cada peso ($\partial L / \partial w$) com backpropagation (regra da cadeia). Se o gradiente é positivo, aumentar o peso aumenta a loss, então o peso deve diminuir; se é negativo, deve aumentar.

### 2. Como descobrir quanto esse peso deve mudar?

> Pelo tamanho do gradiente multiplicado pela **learning rate**: $w \leftarrow w - \eta \cdot \partial L / \partial w$. Learning rate alta dá passos grandes (pode divergir); baixa dá passos pequenos (aprende devagar).

### 3. Como saber se as mudanças realizadas estão realmente melhorando o modelo?

> Recalculando a loss depois de cada atualização (a cada batch/época) e acompanhando se ela cai. Para saber se o modelo melhora de verdade, e não só decora o treino, a loss e as métricas também precisam cair em dados de **validação**.

Não é necessário saber as respostas ainda.

Na próxima aula, vamos descobrir como uma rede neural **aprende com seus próprios erros**.

$$
\boxed{
\text{Loss}
\rightarrow
\text{Gradientes}
\rightarrow
\text{Backpropagation}
\rightarrow
\text{Gradient Descent}
\rightarrow
\text{Atualização dos parâmetros}
}
$$

E estudaremos conceitos como **learning rate**, **épocas**, **batches** e o otimizador **Adam**.

---

# Checklist de aprendizagem

Antes de encerrar, marque o que você consegue explicar sem consultar o notebook:

- [ ] O que é um neurônio artificial.
- [ ] O papel de pesos e bias.
- [ ] O que representa \(z=Wx+b\).
- [ ] Por que utilizamos funções de ativação.
- [ ] Quando utilizar ReLU.
- [ ] Por que utilizamos sigmoid na saída deste problema.
- [ ] O que significa `Dense(4)`.
- [ ] Como calcular o número de parâmetros de uma camada Dense.
- [ ] O que é forward pass.
- [ ] Diferença entre probabilidade e classe.
- [ ] O papel do threshold.
- [ ] O que é uma função de loss.
- [ ] Por que a rede ainda não está aprendendo neste notebook.

## Próxima aula

**Como uma rede neural aprende?**
